# Local tests

Este notebook executa o fluxo principal diretamente:

1. Carrega documentos da pasta `data/`.
2. Cria o índice com `RAG`.
3. Faz perguntas ao chat engine.
4. Gera o PDF com `generate_technical_report`.


In [ ]:
from pathlib import Path
import sys
import uuid

# Detecta a raiz do projeto quando o notebook roda a partir de notebooks/
project_root = Path.cwd().resolve()
if not (project_root / "src").exists() and (project_root.parent / "src").exists():
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from llama_index.core import SimpleDirectoryReader
from src.engine import RAG
from src.report import generate_technical_report

print(f"Project root: {project_root}")

In [ ]:
data_dir = project_root / "data"
candidate_dirs = sorted(data_dir.glob("temp_*"))
docs_dir = next((d for d in candidate_dirs if d.is_dir() and any(d.iterdir())), None)

if docs_dir is None:
    docs_dir = data_dir / f"temp_notebook_{uuid.uuid4()}"
    docs_dir.mkdir(parents=True, exist_ok=True)
    (docs_dir / "sample_requisitos.txt").write_text(
        "Projeto Clarus: sistema de analise tecnica com upload de documentos, "
        "extracao de riscos e geracao de recomendacoes em portugues.",
        encoding="utf-8",
    )

print(f"Pasta de documentos: {docs_dir}")
documents = SimpleDirectoryReader(str(docs_dir)).load_data()
print(f"Documentos carregados: {len(documents)}")

In [ ]:
try:
    rag = RAG()
    rag.create_index(documents)
    print("Indice criado com sucesso.")
except Exception as exc:
    raise RuntimeError(
        "Falha ao criar o indice. Verifique se o Ollama esta ativo e se as dependencias estao instaladas."
    ) from exc

In [ ]:
pergunta = "Quais sao os principais riscos identificados nos documentos?"
resposta = rag.query(pergunta)

print("Pergunta:", pergunta)
print("\nResposta:\n")
print(resposta)

In [ ]:
pergunta = "Quais são os principais requisitos desse documento?"
resposta = rag.query(pergunta)

print("Pergunta:", pergunta)
print("\nResposta:\n")
print(resposta)

In [ ]:
insights_finais = rag.query(
    "Com base no conteudo analisado, gere um resumo com principais insights e conclusoes tecnicas."
)

chat_history = [
    {"role": "user", "content": pergunta},
    {"role": "assistant", "content": resposta},
]

doc_names = [p.name for p in docs_dir.iterdir() if p.is_file()]
pdf_bytes = generate_technical_report(chat_history, doc_names, insights_finais)

output_pdf = project_root / "data" / "relatorio_teste_notebook.pdf"
output_pdf.write_bytes(pdf_bytes)

print(f"PDF gerado em: {output_pdf}")
print(f"Tamanho do arquivo: {len(pdf_bytes)} bytes")